# Suite VLGRD — Reading Verilog with slang

The standards read source text through slang, a complete SystemVerilog front end: the conformance source reads to
a valid tree that prints and reads back the same; each construct reads to its kinds; directives, comments and macro
uses are kept where items and statements are listed; and what slang reports, the reader refuses or a standard lacks
fails with its line and column.

In [ ]:
import os
import tempfile

from mbse.Programs.Framework import Errors
from mbse.Programs.Framework.Syntax import same, walk
from mbse.Programs.Verilog import SystemVerilog2017, SystemVerilog2023, Syntax as S, Verilog2005, VerilogStandard
from mbse.Schemas.Framework import JSON
from support import raises

L = S.LANGUAGE
SOURCES = "../../conformance/sources"


def read(text, **options):
    return SystemVerilog2023.parse(text, **options)


def items(text):
    return read(f"module m;\n{text}\nendmodule\n").items[0].items


def statements(text):
    return read(f"module m;\ninitial begin\n{text}\nend\nendmodule\n").items[0].items[0].body.items


def expression(text):
    return statements(f"x = {text};")[0].value


def fails(text, message, error=Errors.ParseError, **options):
    with raises(error, match=message):
        read(text, **options)

## VLGRD-01 · The conformance source reads to a valid tree that prints and reads back the same

In [ ]:
text = open(f"{SOURCES}/systemverilog2023.sv", encoding="utf-8").read()
tree = read(text, include_paths=[SOURCES])
assert L.validate(tree) == [] and {type(n).__name__ for n in walk(tree)} == set(L.kinds())
printed = SystemVerilog2023.print(tree)
again = read(printed, include_paths=[SOURCES])
assert same(again, tree) and SystemVerilog2023.print(again) == printed
print(len(list(walk(tree))), "syntax nodes")

## VLGRD-02 · Design units: headers, imports, parameters, ANSI, interface and non-ANSI ports

In [ ]:
unit = read("""package p; endpackage
module automatic m import p::*; #(parameter int W = 8, localparam N = 2, parameter type T = logic) (
    input wire [W-1:0] a, output var logic b = 1'b0, inout tri c, ref int r, bus.mp port, interface generic [2]);
endmodule : m
macromodule n (a, b); input a; output reg [3:0] b; endmodule
interface automatic i; endinterface
program static q; endprogram
""")
m = unit.items[1]
assert (m.keyword, m.lifetime, m.name.spelling, m.labeled) == ("module", "automatic", "m", True)
assert m.imports[0].items[0].package.spelling == "p" and m.imports[0].items[0].name is None
parameter, local, kind = m.parameters
assert (parameter.keyword, parameter.type.keyword, parameter.assignments[0].name.spelling) == ("parameter", "int", "W")
assert local.keyword == "localparam" and local.type is None and local.assignments[0].value.spelling == "2"
assert isinstance(kind, S.TypeParameterDeclaration) and kind.assignments[0].type.keyword == "logic"
a, b, c, r, port, generic = m.ports
assert (a.direction, a.net_type, a.type.dimensions[0].left.left.name.spelling) == ("input", "wire", "W")
assert (b.direction, b.var, b.type.keyword, b.value.spelling) == ("output", True, "logic", "1'b0")
assert (c.net_type, r.direction, r.type.keyword) == ("tri", "ref", "int")
assert isinstance(port, S.InterfacePort) and (port.interface.spelling, port.modport.spelling) == ("bus", "mp")
assert generic.interface is None and generic.modport is None and generic.dimensions[0].size.spelling == "2"
n = unit.items[2]
assert n.keyword == "macromodule" and [p.name.spelling for p in n.ports] == ["a", "b"]
assert isinstance(n.items[0], S.PortDeclaration) and n.items[0].declarators[0].name.spelling == "a"
assert n.items[1].type.keyword == "reg" and n.items[1].direction == "output"
assert unit.items[3].lifetime == "automatic" and unit.items[4].lifetime == "static"
bare = read("module k #(int W = 8, type T = logic) (); endmodule\n").items[0].parameters
assert bare[0].keyword is None and bare[0].type.keyword == "int" and bare[1].keyword is None and L.validate(bare[0]) == []
print("ok")

## VLGRD-03 · Data types and dimensions

In [ ]:
declared = items("""logic signed [7:0] a [4][0:1];
int unsigned b [string];
byte c [*];
shortreal d [$:8];
string e [];
p::t f;
struct packed signed { bit [3:0] x, y; } g;
union { int i; real r; } h;
enum { A, B = 2 } k;
event l;
realtime m;
chandle n;""")
a = declared[0]
assert (a.type.keyword, a.type.signing, a.type.dimensions[0].right.spelling) == ("logic", "signed", "0")
assert a.declarators[0].dimensions[0].size.spelling == "4" and isinstance(a.declarators[0].dimensions[1], S.RangeDimension)
assert declared[1].type.signing == "unsigned" and declared[1].declarators[0].dimensions[0].type.keyword == "string"
assert declared[2].declarators[0].dimensions[0].type is None
assert declared[3].type.keyword == "shortreal" and declared[3].declarators[0].dimensions[0].bound.spelling == "8"
assert isinstance(declared[4].declarators[0].dimensions[0], S.UnsizedDimension) and declared[4].type.keyword == "string"
assert (declared[5].type.name.scope.spelling, declared[5].type.name.name.spelling) == ("p", "t")
g = declared[6].type
assert (g.keyword, g.packed, g.signing, [d.name.spelling for d in g.members[0].declarators]) == ("struct", True, "signed", ["x", "y"])
assert declared[7].type.keyword == "union" and not declared[7].type.packed
k = declared[8].type
assert k.base is None and [m.name.spelling for m in k.members] == ["A", "B"] and k.members[1].value.spelling == "2"
assert [d.type.keyword for d in declared[9:]] == ["event", "realtime", "chandle"]
print("ok")

## VLGRD-04 · Declarations and items: nets, variables, typedefs, genvars, imports, modports, assigns and constructs

In [ ]:
declared = items("""wire [1:0] #3 w = 2'b01, v;
const var static int x = 1;
typedef logic [3:0] nibble_t [2];
genvar g, h;
import p::a, p::*;
assign #1 w = v, v = 0;
always @* x = 1;
always_comb x = 2;
always_latch x = 3;
initial x = 4;
final x = 5;
function automatic void f(); endfunction : f
function g2(input a); return a; endfunction
task t(output int o, input logic [1:0] i = 2'b0); endtask
task old; input a; endtask""")
wire, var, typedef, genvar, imports, assign = declared[:6]
assert (wire.net_type, wire.delay.value.spelling, wire.type.dimensions[0].left.spelling) == ("wire", "3", "1")
assert [d.name.spelling for d in wire.declarators] == ["w", "v"] and wire.declarators[0].value.spelling == "2'b01"
assert (var.const, var.var, var.lifetime) == (True, True, "static")
assert typedef.dimensions[0].size.spelling == "2" and [n.spelling for n in genvar.names] == ["g", "h"]
assert [(i.package.spelling, i.name and i.name.spelling) for i in imports.items] == [("p", "a"), ("p", None)]
assert assign.delay.value.spelling == "1" and len(assign.assignments) == 2
assert [c.keyword for c in declared[6:9]] == ["always", "always_comb", "always_latch"]
assert isinstance(declared[9], S.InitialConstruct) and isinstance(declared[10], S.FinalConstruct)
f, g2, t, old = declared[11:]
assert (f.lifetime, f.type.keyword, f.labeled, f.ports) == ("automatic", "void", True, [])
assert g2.type is None and g2.ports[0].direction == "input" and isinstance(g2.body[0], S.ReturnStatement)
assert t.ports[1].value.spelling == "2'b0" and t.ports[0].type.keyword == "int"
assert old.ports == [] and isinstance(old.body[0], S.PortDeclaration)
interface = read("interface i; modport mp (input a, output b, inout c); endinterface").items[0]
assert [(p.direction, p.name.spelling) for p in interface.items[0].items[0].ports] == [("input", "a"), ("output", "b"),
                                                                                      ("inout", "c")]
print("ok")

## VLGRD-05 · Generate constructs and instantiation

In [ ]:
declared = items("""generate for (genvar i = 0; i < 2; i++) begin : g end endgenerate
for (j = 0; j < 2; j = j + 1) assign a[j] = 0;
if (W) begin end else if (V) assign a = 1; else begin : e end : e
case (W) 1, 2: assign a = 1; default: begin end endcase
sub #(1, .N(2), .T(logic)) u1 (a, .b(c), .d(), .e), u2[1:0] (.*);""")
region = declared[0]
loop = region.items[0]
assert loop.genvar and loop.name.spelling == "i" and isinstance(loop.step, S.IncrementExpression) and loop.body.name.spelling == "g"
assert not declared[1].genvar and isinstance(declared[1].body, S.ContinuousAssign)
conditional = declared[2]
assert isinstance(conditional.alternative, S.GenerateIf) and conditional.alternative.alternative.labeled
case = declared[3]
assert [len(i.expressions) for i in case.items] == [2, 0]
instantiation = declared[4]
assert [type(p).__name__ for p in instantiation.parameters] == ["IntegerLiteral", "NamedConnection", "NamedConnection"]
assert instantiation.parameters[2].value.keyword == "logic"
u1, u2 = instantiation.instances
assert [type(c).__name__ for c in u1.connections] == ["NameExpression", "NamedConnection", "NamedConnection", "NamedConnection"]
assert u1.connections[2].value is None and not u1.connections[2].implicit and u1.connections[3].implicit
assert isinstance(u2.connections[0], S.WildcardConnection) and u2.dimensions[0].left.spelling == "1"
print("ok")

## VLGRD-06 · Statements

In [ ]:
body = statements("""a = 1; b <= #2 c; d += 1; e <= @(posedge clk) f;
f(); i++; --j; ;
begin : named int k; end : named
fork join_any
fork : par join_none
unique if (a) b = 1; else if (c) b = 2; else b = 3;
priority casez (a) inside 1, [2:3]: ; default b = 0; endcase
casex (a) 1: ; endcase
for (int k = 0, m = 1; k < 2; k++, m--) ;
for (;;) break;
while (a) continue;
repeat (2) ;
forever ;
do a = 1; while (b);
foreach (q[i, j]) a = 1;
#1 ;
@(a or posedge b, negedge c iff d) ;
@e a = 1;
@(*) ;
wait (a) ;
-> ev;
->> ev;
disable named;
disable fork;
assert (a) b = 1; else b = 2;
assume #0 (a);
cover final (a) b = 1;
return;""")
assignment = body[0]
assert (assignment.operator, assignment.timing) == ("=", None)
assert body[1].operator == "<=" and body[1].timing.value.spelling == "2"
assert body[2].operator == "+=" and body[3].timing.events[0].edge == "posedge"
assert isinstance(body[4].expression, S.CallExpression) and body[5].expression.postfix and not body[6].expression.postfix
assert isinstance(body[7], S.NullStatement)
named = body[8]
assert named.name.spelling == "named" and named.labeled and isinstance(named.items[0], S.VariableDeclaration)
assert body[9].join == "join_any" and body[10].name.spelling == "par" and body[10].join == "join_none"
conditional = body[11]
assert conditional.qualifier == "unique" and conditional.alternative.qualifier is None
assert isinstance(conditional.alternative.alternative, S.AssignmentStatement)
case = body[12]
assert (case.qualifier, case.keyword, case.inside) == ("priority", "casez", True)
assert isinstance(case.items[0].expressions[1], S.ValueRange) and case.items[1].expressions == []
assert body[13].keyword == "casex"
loop = body[14]
assert [type(i).__name__ for i in loop.initializers] == ["VariableDeclaration", "VariableDeclaration"] and len(loop.steps) == 2
assert loop.initializers[0].type.keyword == "int" and loop.initializers[1].type is None  # `m` is an int too
assert body[15].condition is None and body[15].initializers == []
assert [type(s).__name__ for s in body[16:21]] == ["WhileStatement", "RepeatStatement", "ForeverStatement",
                                                   "DoWhileStatement", "ForeachStatement"]
assert [v.spelling for v in body[20].variables] == ["i", "j"]
assert body[21].body is None and body[21].timing.value.spelling == "1"
events = body[22].timing.events
assert [(e.edge, e.expression.name.spelling) for e in events] == [(None, "a"), ("posedge", "b"), ("negedge", "c")]
assert events[2].condition.name.spelling == "d"
assert body[23].timing.events[0].expression.name.spelling == "e" and body[24].timing.events == []
assert body[25].body is None and not body[26].nonblocking and body[27].nonblocking
assert body[28].target.name.spelling == "named" and body[29].target is None
asserted = body[30]
assert asserted.pass_action.operator == "=" and asserted.fail_action.value.spelling == "2"
assert body[31].deferral == "#0" and body[31].keyword == "assume" and body[32].deferral == "final"
assert body[33].value is None
print("ok")

## VLGRD-07 · Expressions

In [ ]:
values = [expression(e) for e in [
    "a + b * c", "(a + b) * c", "-a", "&a", "~^a", "a ? b : c", "a inside {1, [2:3]}", "{a, b}", "{2{a}}",
    "'{1, 2}", "'{default: 0, a: 1, int: 2}", "t'{1}", "f(1, .b(2))", "$clog2(1)", "$time", "q.size()",
    "int'(x)", "8'(x)", "signed'(x)", "p::c", "s.m[1][7:0]", "a[i +: 2]", "a[i -: 2]", "q[$]", "8'hFF", "1.5e3",
    "10ns", "'1", "\"s\\n\"", "x++", "(y = 1)", "a <-> b"]]
assert values[0].operator == "+" and values[0].right.operator == "*"
assert isinstance(values[1].left, S.ParenthesizedExpression)
assert [v.operator for v in values[2:5]] == ["-", "&", "~^"]
assert isinstance(values[5], S.ConditionalExpression) and isinstance(values[6].set[1], S.ValueRange)
assert len(values[7].items) == 2 and values[8].count.spelling == "2"
assert values[9].items[1].spelling == "2" and values[9].type is None
assert [(i.key is None, type(i.key).__name__) for i in values[10].items] == [(True, "NoneType"), (False, "NameExpression"),
                                                                           (False, "IntegerAtomType")]
assert values[11].type.name.spelling == "t"
assert isinstance(values[12].arguments[1], S.NamedConnection) and values[13].name == "$clog2"
assert values[14].arguments == [] and values[15].callee.member.spelling == "size"
assert values[16].type.keyword == "int" and values[17].type.spelling == "8" and values[18].type.signing == "signed"
assert values[19].name.scope.spelling == "p"
member = values[20]
assert isinstance(member, S.RangeSelect) and isinstance(member.value, S.IndexExpression)
assert member.value.value.member.spelling == "m"
assert values[21].operator == "+:" and values[22].operator == "-:" and isinstance(values[23].index, S.DollarExpression)
assert [type(v).__name__ for v in values[24:29]] == ["IntegerLiteral", "RealLiteral", "TimeLiteral", "UnbasedUnsizedLiteral",
                                                     "StringLiteral"]
assert values[24].spelling == "8'hFF" and values[27].value == "1" and values[28].text == "s\\n"
assert values[29].postfix and isinstance(values[30].expression, S.AssignmentExpression) and values[31].operator == "<->"
print("ok")

## VLGRD-08 · Directives and comments where items and statements are listed; conditional compilation as a tree

In [ ]:
unit = read("""// one
`resetall
`define A 1
`define F(x, y) ((x) + (y))
`undef A
`default_nettype wire
`timescale 10ns / 1ns
`ifndef B
module m; /* inner */
`ifdef C
wire c;
`elsif D
wire d;
`else
wire e; // trailing
`endif
initial begin
`ifdef C
x = 1;
`endif
end
endmodule
`endif
""")
comment, resetall, define, function, undef, nettype, timescale, ifndef = unit.items
assert comment.text == " one" and not comment.trailing and resetall.text == "`resetall"
assert (define.name.spelling, define.body, define.function_like) == ("A", "1", False)
assert function.function_like and [p.spelling for p in function.parameters] == ["x", "y"]
assert undef.name.spelling == "A" and nettype.net_type == "wire" and (timescale.unit, timescale.precision) == ("10ns", "1ns")
assert ifndef.negated and ifndef.name.spelling == "B" and not ifndef.has_else
module = ifndef.items[0]
inner, ifdef = module.items[:2]
assert inner.block and inner.trailing
assert isinstance(ifdef.items[0], S.DisabledText) and ifdef.items[0].text == "wire c;"
assert isinstance(ifdef.branches[0].items[0], S.DisabledText) and ifdef.has_else
assert isinstance(ifdef.alternative[0], S.NetDeclaration) and ifdef.alternative[1].trailing
statement_ifdef = module.items[2].body.items[0]
assert isinstance(statement_ifdef, S.IfdefDirective) and statement_ifdef.items[0].text == "x = 1;"
taken = read("`ifdef C\nmodule m; endmodule\n`endif\n", defines=["C"])
assert isinstance(taken.items[0].items[0], S.ModuleDeclaration)
assert SystemVerilog2023.print(unit).startswith("// one\n`resetall\n`define A 1\n`define F(x, y) ((x) + (y))")
print("ok")

## VLGRD-09 · Macro uses and included files

In [ ]:
unit = read("""`define W 8
`define SUM(a, b) a + b
module m; wire [`W-1:0] x = `SUM(1, 2); endmodule
""")
declaration = unit.items[2].items[0]
width = declaration.type.dimensions[0].left.left
assert isinstance(width, S.MacroUsage) and width.name.spelling == "W" and width.arguments is None
value = declaration.declarators[0].value
assert isinstance(value, S.MacroUsage) and value.arguments == "1, 2"
with tempfile.TemporaryDirectory() as folder:
    with open(os.path.join(folder, "inc.svh"), "w") as f:
        f.write("`define FROM_INCLUDE 1\nwire included;\n")
    included = read('module m;\n`include "inc.svh"\nwire own;\nendmodule\n', include_paths=[folder])
    assert [type(i).__name__ for i in included.items[0].items] == ["IncludeDirective", "NetDeclaration"]
    assert included.items[0].items[1].declarators[0].name.spelling == "own"
fails('`include "missing.svh"\nmodule m; endmodule\n', "could not open include file")
print("ok")

## VLGRD-10 · What slang reports, what the reader does not support and what a standard lacks fail at their place

In [ ]:
fails("module m;\n  wire = ;\nendmodule\n", "line 2")
fails("`define S a = 1;\nmodule m; initial begin `S end endmodule\n", "a macro use that is not a whole expression")
fails("config cfg; design work.top; endconfig\n", "unsupported syntax: ConfigDeclaration")
fails("module m; wire [3:0] a = 1 + `undefined_macro; endmodule\n", "unknown directive")
with raises(Errors.ParseError, match="line 1, column 1: PackageDeclaration is not Verilog, but this is Verilog-2005"):
    Verilog2005.parse("package p; endpackage\n")
assert isinstance(Verilog2005.parse("module m (input a); endmodule\n").items[0], S.ModuleDeclaration)
error = None
try:
    read("module m;\n\n   wire = ;\nendmodule\n")
except Errors.ParseError as caught:
    error = caught
assert error is not None and (error.line, error.column) == (3, 9), (error.line, error.column)
print("ok")

## VLGRD-11 · What is outside the kinds is refused, naming it, where it is

In [ ]:
for source, message in [
    ("`define N n\nmodule m; wire `N; endmodule", "line 2, column 16: unsupported syntax: a macro use that is not a whole expression"),
    ("`ifdef A\nmodule a;\n`else\nmodule b;\n`endif\nendmodule", "line 5, column 1: unexpected `endif"),
    ("module m; enum { A[1][2] } e; endmodule", "line 1, column 18: unsupported syntax: Declarator"),
    ("module m; enum { A[] } e; endmodule", "line 1, column 18: unsupported syntax: Declarator"),
    ("module m; wire [3+:2] a; endmodule", "unsupported syntax: AscendingRangeSelect"),
    ("module m; $error(int); endmodule", "line 1, column 11: unsupported syntax: ElabSystemTask"),
    ("module m; int [1:0] a; endmodule", "unsupported syntax: IntType"),
    ("module m; initial #1step ; endmodule", "unsupported syntax: OneStepDelay"),
    ("`pragma protect begin_protected\n`pragma protect data_block\nabc\n`pragma protect end_protected\nmodule m; endmodule",
     "unsupported syntax: skipped text"),
    ("module a;\n`ifdef A\n`else\nendmodule\n`endif", "line 3, column 1: expected `endif"),
    ("module m; typedef a.b.c t; endmodule", "unsupported syntax: ScopedName"),
    ("module m; initial x = a.b#(1); endmodule", "unsupported syntax: ScopedName"),
    ("class c; local typedef class d; endclass", "unsupported syntax: ClassPropertyDeclaration"),
    ("module m; initial x = f(#1 a); endmodule", "line 1, column 25: unsupported syntax: TimingControlExpression"),
    ("module m; sequence s; (a[*2], x = b[*2]); endsequence endmodule", "line 1, column 31: unsupported syntax: SimpleSequenceExpr"),
    ("module m; assert property (b) $display(a[*2]); endmodule", "line 1, column 40: unsupported syntax: SimpleSequenceExpr"),
    ("interface i; modport mp ((* a *) input d); endinterface", "line 1, column 26: unsupported syntax: AttributeInstance"),
    ("module m ((* a *) .p(x)); endmodule", "line 1, column 11: unsupported syntax: AttributeInstance"),
    ("primitive p (.*); table 0 : 1; endtable endprimitive", "line 1, column 13: unsupported syntax: WildcardUdpPortList"),
    ("class c; function void f(); ok = obj.randomize() with (a.b) { x > 0; }; endfunction endclass",
     "unsupported syntax: ScopedName"),
    ("class c; function void f(); x = local::p::y; endfunction endclass", "unsupported syntax: ScopedName"),
    ("module m; property p; (not a, x = 1); endproperty endmodule", "unsupported syntax: ParenthesizedPropertyExpr"),
]:
    fails(source + "\n", message)
print("ok")

## VLGRD-12 · Corner cases the kinds hold

In [ ]:
unit = read('''// é: positions count characters, not bytes
module m #(parameter type T, parameter W) ();
logic a [int]; logic b [t]; automatic int c;
function static int f(); endfunction
if (1) begin end
u #(.N()) i(); u j();
initial begin
    for (i = 0, j = 1; i < 2; i++) ;
    @(posedge a or b iff c) ;
    @(e.triggered) ;
    x = a.b[1].c + a.b.c + t::a;
    x = '{a, b} + q.sum() + int'(1) + signed'(1) + a.b();
    x = $bits(int) + $bits(logic [1:0]) + f()[0] + f(a).b + f(.a(), .b(c));
    x: begin end
    for (int k; k < 2; k++) ;
    assert (a);
    x = """multi
line""";
end
if (1) g: begin end
endmodule
''')
module = unit.items[1]
assert unit.items[0].text == " é: positions count characters, not bytes"
assert module.parameters[0].assignments[0].type is None and module.parameters[1].assignments[0].value is None
assert module.items[0].declarators[0].dimensions[0].type.keyword == "int"
assert module.items[1].declarators[0].dimensions[0].size.name.spelling == "t"
assert module.items[2].lifetime == "automatic" and module.items[3].lifetime == "static"
assert module.items[5].parameters[0].value is None and module.items[6].instances[0].connections == []
body = module.items[7].body.items
assert len(body[0].initializers) == 2 and body[1].timing.events[1].condition.name.spelling == "c"
assert [type(a).__name__ for a in body[5].value.left.left.left.left.arguments] == ["IntegerAtomType"]
assert body[6].name.spelling == "x" and body[7].initializers[0].declarators[0].value is None
assert module.items[8].consequence.name.spelling == "g"
assert body[8].pass_action is None and body[8].fail_action is None
string = body[9].value
assert string.triple and string.text == "multi\nline"
with raises(Errors.ParseError, match="StringLiteral.triple needs SystemVerilog-2023, but this is SystemVerilog-2017"):
    SystemVerilog2017.parse('module m; initial x = """s"""; endmodule\n')
assert same(read(SystemVerilog2023.print(unit)), unit)
fails("// é\nmodule m; wire = ; endmodule\n", "line 2, column 16")
print("ok")


## VLGRD-13 · Reading for the other implementation: a request in, a tree or an error out, as JSON

In [ ]:
import json
import subprocess
import sys

from mbse.Programs.Verilog import read as reader

request = {"text": "module m; wire a; endmodule\n", "year": 2023, "family": "SystemVerilog", "include_paths": [], "defines": []}
response = json.loads(reader.respond(json.dumps(request)))
assert list(response) == ["tree"]
tree = JSON.FromJSON(S.LANGUAGE.Builders).Reachable(S.SourceText.Schema, response["tree"])
assert same(tree, read(request["text"]))
failed = json.loads(reader.respond(json.dumps({**request, "text": "module m;\n wire = ;\nendmodule\n"})))
assert failed == {"error": "expected declarator", "line": 2, "column": 7}, failed
lacking = json.loads(reader.respond(json.dumps({**request, "year": 2005, "family": "Verilog", "text": "package p; endpackage\n"})))
assert lacking["error"] == "PackageDeclaration is not Verilog, but this is Verilog-2005"
run = subprocess.run([sys.executable, "-m", "mbse.Programs.Verilog.read"], input=json.dumps(request).encode("utf-8"),
                     capture_output=True, check=True)
assert json.loads(run.stdout.decode("utf-8")) == response
print("ok")


## VLGRD-14 · Classes, their properties and methods, and objects

In [ ]:
unit = read('''typedef class fwd;
typedef plain;
typedef interface class ic;
virtual class base #(type T = int, int N = 2) extends parent #(T) implements i1, i2;
  rand bit [3:0] a; randc int b; local static const int c = 1; protected int d;
  virtual bus_if.mp vif; virtual interface bus_if #(4) v2; c #(8, .T(int)) other;
  extern virtual function void f(int x);
  pure virtual task t();
  extern protected static function int g();
  function new(int x = 0); super.new(x); this.a = x; x = new; x = new y; d = new[4](d); d = new[2];
    x = null; x = p::q::r; x = c#(1)::new(3); x = c#(1)::s; super.g(); endfunction
  local virtual task run(); endtask
endclass : base
class derived extends base #(byte, 3)(1, .x(2)); endclass
interface class i1 extends j, k; pure virtual function void h(); endclass
''')
forward, plain, forward_interface, base, derived, interface = unit.items
assert plain.keyword is None and plain.name.spelling == "plain"
assert forward.keyword == "class" and forward_interface.keyword == "interface class" and forward.name.spelling == "fwd"
assert (base.virtual, base.interface, base.labeled, base.name.spelling) == (True, False, True, "base")
assert [type(p).__name__ for p in base.parameters] == ["TypeParameterDeclaration", "ParameterDeclaration"]
assert base.parameters[0].keyword is None and base.parameters[1].type.keyword == "int"
assert isinstance(base.base.name, S.ParameterizedName) and base.base.name.parameters[0].name.spelling == "T"
assert [i.name.spelling for i in base.interfaces] == ["i1", "i2"] and base.arguments == []
a, b, c, d, vif, v2, other, f, t, g, constructor, run = base.items
assert (a.random, b.random, c.visibility, c.lifetime, c.const, d.visibility) == ("rand", "randc", "local", "static", True,
                                                                               "protected")
assert (vif.type.interface.spelling, vif.type.modport.spelling, vif.type.interface_keyword) == ("bus_if", "mp", False)
assert v2.type.interface_keyword and v2.type.parameters[0].spelling == "4" and v2.type.modport is None
assert [type(p).__name__ for p in other.type.name.parameters] == ["IntegerLiteral", "NamedConnection"]
assert (f.extern, f.virtual, f.pure, f.body) == (True, True, False, []) and (t.pure, t.virtual) == (True, True)
assert (g.extern, g.visibility, g.static, g.lifetime) == (True, "protected", True, None)
assert constructor.name.spelling == "new" and constructor.type is None and (run.visibility, run.virtual) == ("local", True)
body = [s.value if isinstance(s, S.AssignmentStatement) else s.expression for s in constructor.body]
assert isinstance(body[0], S.NewExpression) and isinstance(body[0].scope, S.SuperExpression)
assert isinstance(body[1], S.NameExpression) and isinstance(constructor.body[1].target.value, S.ThisExpression)
assert body[2].scope is None and body[2].arguments == [] and body[3].value.name.spelling == "y"
assert body[4].size.spelling == "4" and body[4].value.name.spelling == "d" and body[5].value is None
assert isinstance(body[6], S.NullLiteral)
nested = body[7].name
assert (nested.scope.spelling, nested.name.scope.spelling, nested.name.name.spelling) == ("p", "q", "r")
assert body[8].scope.name.spelling == "c" and body[8].arguments[0].spelling == "3"
assert isinstance(body[9].name.scope, S.ParameterizedName) and body[9].name.name.spelling == "s"
assert isinstance(body[10].callee.value, S.SuperExpression)
assert derived.base.name.parameters[0].keyword == "byte" and len(derived.arguments) == 2
assert interface.interface and [i.name.spelling for i in interface.interfaces] == ["j", "k"]
assert same(read(SystemVerilog2023.print(unit)), unit)
print("ok")


## VLGRD-15 · Constraints, randomization with inline constraints, array methods with `with`, `randcase` and `void'`

In [ ]:
unit = read('''class c;
  rand int x, y; rand int q[];
  constraint k { // first
    x > 0; soft y < 10; x -> { y == 1; } x -> y == 2; if (x) y > 2; else { y < 3; } foreach (q[i]) q[i] < 5;
    solve x before y, q; disable soft y; unique { x, y, [1:2] }; x dist { 1 := 2, [3:4] :/ 5, 7, default :/ 1 }; }
  static constraint s { }
  extern constraint e;
  pure constraint p;
  static constraint plain;
  function void f();
    ok = this.randomize() with { x < local::y; }; ok = obj.randomize(x, y) with (x) { x > 0; };
    ok = std::randomize(a, b) with { a < b; }; ok = obj.randomize() with () { /* none */ };
    r = q.find(item) with (item > 1); q.sort with (item.x);
    randcase 1: a = 1; x + 1: begin end endcase
    void'(obj.randomize());
  endfunction
endclass
constraint c::e { x != y; }
''')
c, out_of_block = unit.items
k, s, e, p, plain = c.items[2:7]
assert k.name.spelling == "k" and not k.static and k.items[0].text == " first" and k.items[0].trailing
constraints = k.items[1:]
assert [type(i).__name__ for i in constraints] == [
    "ExpressionConstraint", "ExpressionConstraint", "ImplicationConstraint", "ImplicationConstraint",
    "ConditionalConstraint", "ForeachConstraint", "SolveBeforeConstraint", "DisableSoftConstraint", "UniqueConstraint",
    "ExpressionConstraint"]
assert not constraints[0].soft and constraints[1].soft
assert isinstance(constraints[2].body, S.ConstraintBlock) and isinstance(constraints[3].body, S.ExpressionConstraint)
assert isinstance(constraints[4].consequence, S.ExpressionConstraint) and isinstance(constraints[4].alternative, S.ConstraintBlock)
assert [v.spelling for v in constraints[5].variables] == ["i"] and constraints[5].array.name.spelling == "q"
assert [e.name.spelling for e in constraints[6].solve] == ["x"] and [e.name.spelling for e in constraints[6].before] == ["y", "q"]
assert constraints[7].target.name.spelling == "y" and isinstance(constraints[8].set[2], S.ValueRange)
dist = constraints[9].expression
assert isinstance(dist, S.DistExpression) and dist.value.name.spelling == "x"
assert [(type(i.value).__name__, i.operator) for i in dist.items] == [("IntegerLiteral", ":="), ("ValueRange", ":/"),
                                                                      ("IntegerLiteral", None), ("NoneType", ":/")]
assert s.static and s.items == [] and (e.qualifier, p.qualifier, plain.qualifier, plain.static) == ("extern", "pure", None, True)
body = c.items[7].body
calls = [statement.value for statement in body[:5]]
assert isinstance(calls[0], S.RandomizeWithExpression) and not calls[0].restricted
assert isinstance(calls[0].items[0].expression.right.name, S.LocalName)
assert calls[1].restricted and [v.spelling for v in calls[1].variables] == ["x"] and len(calls[1].call.arguments) == 2
assert calls[2].call.callee.name.scope.spelling == "std"
assert calls[3].restricted and calls[3].variables == [] and calls[3].items[0].block
assert isinstance(calls[4], S.ArrayMethodWithExpression) and calls[4].expression.operator == ">"
assert isinstance(body[5].expression, S.ArrayMethodWithExpression) and isinstance(body[5].expression.call, S.MemberExpression)
randcase = body[6]
assert [i.weight.spelling if isinstance(i.weight, S.IntegerLiteral) else i.weight.operator for i in randcase.items] == ["1", "+"]
assert body[7].expression.type.keyword == "void" and isinstance(body[7].expression.value, S.CallExpression)
assert out_of_block.name.scope.spelling == "c" and out_of_block.items[0].expression.operator == "!="
assert same(read(SystemVerilog2023.print(unit)), unit)
print("ok")


## VLGRD-16 · Assertions: properties, sequences, clocking blocks, cycle delays, `let` and labels

In [ ]:
unit = read('''module m;
  default clocking cb @(posedge clk); default input #1step output #0; input #1 a, b; output negedge c; inout d = top.x;
    input #2 output #3 e; endclocking : cb
  global clocking gc @(posedge fast); endclocking
  default clocking @(negedge clk); property inner; if (a) b; endproperty endclocking
  default clocking cb;
  default disable iff (rst);
  sequence s1(x, int y = 1, local input logic z); int v; @(posedge clk) (a, v = x) ##1 b[*2] ##[1:3] c[->1] ##[*] d[=2:3]
    ##[+] e[*] ##1 f[+] ##(y) g; endsequence : s1
  sequence s2; ##1 a ##2 (b ##1 c)[*3] and first_match(d ##1 e, v++) or f intersect g within h throughout i; endsequence
  property p1(sequence q, untyped u); disable iff (rst) q |-> not (a until b) and s_eventually [1:$] c; endproperty
  property p2; if (a) strong(b ##1 c) else weak(d) implies nexttime [2] e iff always [1:3] f; endproperty
  property p3; accept_on (x) a s_until b or sync_reject_on (y) c until_with d; endproperty
  property p4; case (m) 0, 1: a |=> b; default: (c #=# d); endcase; endproperty
  a1: assert property (@(posedge clk) disable iff (rst) a |=> b) else $error("x");
  cover sequence (s1(1));
  restrict property (a ##1 @(negedge clk) b);
  assume property (s_nexttime a or @(negedge clk) b |-> c);
  assert #0 (x);
  let max(a, b = 1) = a > b ? a : b;
  initial begin ##2; ##1 cb.c <= 1; cb.c <= ##2 a; chk: assert (x); a2: assert property (p1(s1, 1));
    expect (@(posedge clk) a ##1 b) else $error; lab: x = 1; end
endmodule
''').items[0]
cb, gc, anonymous, default, disable, s1, s2, p1, p2, p3, p4, a1, cover, restrict, assume, deferred, let, initial = unit.items
assert (cb.scope, cb.name.spelling, cb.labeled, gc.scope, anonymous.name) == ("default", "cb", True, "global", None)
assert isinstance(anonymous.items[0], S.PropertyDeclaration) and anonymous.items[0].spec.property.alternative is None
skews, inputs, outputs, inouts, both = cb.items
assert skews.input.delay.value.spelling == "1step" and skews.output.delay.value.spelling == "0"
assert (inputs.direction, [s.name.spelling for s in inputs.signals], inputs.input_skew.delay.value.spelling) == ("input", ["a", "b"], "1")
assert (outputs.direction, outputs.output_skew.edge, outputs.output_skew.delay) == ("output", "negedge", None)
assert inouts.direction == "inout" and inouts.signals[0].value.member.spelling == "x"
assert both.direction == "input output" and both.output_skew.delay.value.spelling == "3"
assert default.name.spelling == "cb" and isinstance(disable.condition, S.ParenthesizedExpression)
assert [(p.local, p.direction, p.type and type(p.type).__name__, p.value and p.value.spelling) for p in s1.ports] == [
    (False, None, None, None), (False, None, "IntegerAtomType", "1"), (True, "input", "IntegerVectorType", None)]
assert s1.labeled and s1.variables[0].declarators[0].name.spelling == "v"
clocked = s1.sequence
assert isinstance(clocked, S.ClockedSequence) and isinstance(clocked.sequence, S.DelaySequence)
chain = clocked.sequence
assert isinstance(chain.first, S.ParenthesizedSequence) and chain.first.items[0].operator == "="
assert [type(s.delay).__name__ for s in chain.steps] == ["IntegerLiteral", "CycleRange", "CycleRange", "CycleRange",
                                                         "IntegerLiteral", "ParenthesizedExpression"]
assert [(s.delay.low.spelling, type(s.delay.high).__name__) for s in chain.steps[2:4]] == [("0", "DollarExpression"),
                                                                                         ("1", "DollarExpression")]
repetitions = [s.sequence for s in chain.steps[:5]]
assert [(r.operator, type(r.count).__name__) for r in repetitions] == [("*", "IntegerLiteral"), ("->", "IntegerLiteral"),
                                                                      ("=", "CycleRange"), ("*", "CycleRange"),
                                                                      ("*", "CycleRange")]
assert (repetitions[3].count.low.spelling, repetitions[4].count.low.spelling) == ("0", "1")
assert s2.sequence.operator == "or" and s2.sequence.left.operator == "and" and s2.sequence.left.left.first is None
assert isinstance(s2.sequence.left.right, S.FirstMatchSequence) and s2.sequence.right.right.right.operator == "throughout"
assert [t.keyword for t in (p1.ports[0].type, p1.ports[1].type)] == ["sequence", "untyped"]
implication = p1.spec.property
assert p1.spec.disable.name.spelling == "rst" and (implication.operator, implication.consequent.operator) == ("|->", "and")
assert implication.consequent.left.operator == "not" and isinstance(implication.consequent.left.operand, S.ParenthesizedProperty)
assert implication.consequent.right.range.high.__class__ is S.DollarExpression
conditional = p2.spec.property
assert isinstance(conditional, S.ConditionalProperty) and conditional.consequence.keyword == "strong"
alternative = conditional.alternative  # `implies` binds looser than `iff`
assert alternative.operator == "implies" and alternative.right.operator == "iff" and alternative.right.left.range.spelling == "2"
abort = p3.spec.property  # `or` binds tighter than `s_until`
assert abort.keyword == "accept_on" and abort.operand.operator == "s_until" and abort.operand.right.right.keyword == "sync_reject_on"
case = p4.spec.property
assert [len(i.expressions) for i in case.items] == [2, 0] and isinstance(case.items[1].body, S.ParenthesizedProperty)
assert a1.label.spelling == "a1" and a1.assertion.spec.clock.events[0].edge == "posedge"
assert a1.assertion.fail_action.expression.name == "$error" and a1.assertion.pass_action is None
assert cover.assertion.sequence and cover.assertion.keyword == "cover" and restrict.assertion.keyword == "restrict"
assert isinstance(restrict.assertion.spec.property.steps[0].sequence, S.ClockedSequence)
assert isinstance(assume.assertion.spec.property.right, S.ClockedProperty)
assert isinstance(deferred.assertion, S.ImmediateAssertion) and deferred.label is None
assert [p.name.spelling for p in let.ports] == ["a", "b"] and isinstance(let.value, S.ConditionalExpression)
body = initial.body.items
assert isinstance(body[0].timing, S.CycleDelay) and body[0].body is None and body[1].body.operator == "<="
assert isinstance(body[2].timing, S.CycleDelay)
assert isinstance(body[3], S.LabeledStatement) and isinstance(body[3].statement, S.ImmediateAssertion)
assert body[4].label.spelling == "a2" and isinstance(body[4].statement, S.ConcurrentAssertion)
assert isinstance(body[5], S.ExpectStatement) and body[5].fail_action is not None
assert body[6].label.spelling == "lab" and isinstance(body[6].statement, S.AssignmentStatement)
assert same(read(SystemVerilog2023.print(unit)).items[0], unit)
print("ok")


## VLGRD-17 · Coverage: covergroups, coverpoints, bins of every kind, crosses and select expressions

In [ ]:
unit = read("""class c;
  covergroup cg(int lo, ref int r) @(posedge clk);
    option.per_instance = 1; type_option.weight = 2;
    cp_a: coverpoint a iff (en) { bins low[4] = {[0:3], 5} iff (x); bins mid = {[4:$]} with (item % 2 == 0);
      wildcard bins w = {4'b1??0}; illegal_bins bad = default; ignore_bins ig[] = (1 => 2 => 3), (4 => 5[*2:3] => 6[->1] => 7[=2]);
      bins tr = (1, 2 => 3); bins d = default sequence; option.at_least = 2; bins s = cp_b with (item > 1); bins e = a + 1; }
    coverpoint b;
    cp_c: coverpoint a + b;
    bit [3:0] cp_t: coverpoint a {}
    x_ab: cross cp_a, b iff (en) { bins both = binsof(cp_a.low) && binsof(b) intersect {[1:2]};
      ignore_bins no = !binsof(cp_a) || binsof(cp_a.mid); bins w = x_ab with (cp_a > 1); bins p = (binsof(b));
      function int f(); endfunction }
    cross cp_a, cp_c;
  endgroup : cg
  covergroup cg2 with function sample(int v); coverpoint v; endgroup
  covergroup cg3; endgroup
endclass
""")
cg, cg2, cg3 = unit.items[0].items
assert [p.name.spelling for p in cg.ports] == ["lo", "r"] and cg.ports[1].direction == "ref" and cg.labeled
assert cg.clock.events[0].edge == "posedge" and cg.sample is None
assert cg2.clock is None and [p.name.spelling for p in cg2.sample.ports] == ["v"] and (cg3.clock, cg3.sample) == (None, None)
per_instance, weight, cp_a, b, cp_c, cp_t, x_ab, cross = cg.items
assert (per_instance.target.value.name.spelling, per_instance.target.member.spelling, per_instance.value.spelling) == (
    "option", "per_instance", "1")
assert weight.target.value.name.spelling == "type_option"
assert (cp_a.label.spelling, cp_a.condition.name.spelling, cp_a.type) == ("cp_a", "en", None)
low, mid, w, bad, ig, tr, d, at_least, s, e = cp_a.items
assert (low.keyword, low.array, low.size.spelling, low.condition.name.spelling) == ("bins", True, "4", "x")
assert [type(v).__name__ for v in low.initializer.values] == ["ValueRange", "IntegerLiteral"]
assert isinstance(mid.initializer.values[0].right, S.DollarExpression) and mid.initializer.filter.operator == "=="
assert w.wildcard and bad.keyword == "illegal_bins" and isinstance(bad.initializer, S.BinsDefault) and not bad.initializer.sequence
assert ig.keyword == "ignore_bins" and ig.array and ig.size is None
first, second = ig.initializer.sequences
assert [[v.spelling for v in step.values] for step in first.steps] == [["1"], ["2"], ["3"]]
assert [(step.operator, type(step.count).__name__) for step in second.steps] == [
    (None, "NoneType"), ("*", "CycleRange"), ("->", "IntegerLiteral"), ("=", "IntegerLiteral")]
assert len(tr.initializer.sequences[0].steps[0].values) == 2 and d.initializer.sequence
assert isinstance(at_least, S.CoverageOption)
assert s.initializer.expression.name.spelling == "cp_b" and s.initializer.filter.operator == ">"
assert e.initializer.expression.operator == "+" and e.initializer.filter is None
assert b.label is None and b.items == [] and cp_c.expression.operator == "+"
assert cp_t.type.keyword == "bit" and cp_t.items == []  # `{}` is written `;`
assert [i.name.spelling for i in x_ab.items] == ["cp_a", "b"] and x_ab.condition.name.spelling == "en"
both, no, filtered, parenthesized, function = x_ab.body
assert isinstance(both.select, S.BinaryBinsSelect) and both.select.operator == "&&"
assert isinstance(both.select.left.target, S.MemberExpression) and both.select.left.intersect == []
assert isinstance(both.select.right.intersect[0], S.ValueRange)
assert isinstance(no.select.left, S.NotBinsSelect) and no.select.operator == "||"
assert isinstance(filtered.select, S.FilteredBinsSelect) and filtered.select.select.name.spelling == "x_ab"
assert isinstance(parenthesized.select, S.ParenthesizedBinsSelect) and isinstance(function, S.FunctionDeclaration)
assert cross.label is None and cross.body == []
assert same(read(SystemVerilog2023.print(unit)), unit)
print("ok")


## VLGRD-18 · Attributes on items, ports, statements, operators and calls

In [ ]:
unit = read('''(* top, version = "1.0" *) module m ((* keep *) input a, output b);
  (* keep *) (* dont_touch = 1 *) wire w;
  (* full_case, parallel_case *) initial (* lab = 2 *) case (a) default: ; endcase
  assign b = a + (* mark *) w ? (* x *) a : b;
  initial begin x = f (* inline *) (1); y = $clog2 (* fast *) (8); z = - (* neg *) a; i (* inc *)++; ++ (* pre *) i;
    lab: (* attr *) x = 1; end
  class c; (* keep *) int p; endclass
endmodule
''')
top = unit.items[0]
assert isinstance(top, S.AttributedItem) and [s.name.spelling for s in top.attributes[0].specs] == ["top", "version"]
assert top.attributes[0].specs[1].value.text == "1.0" and top.attributes[0].specs[0].value is None
module = top.item
assert isinstance(module.ports[0], S.AttributedPort) and module.ports[0].port.direction == "input"
wire, initial, assign, block, cls = module.items
assert [i.specs[0].name.spelling for i in wire.attributes] == ["keep", "dont_touch"] and isinstance(wire.item, S.NetDeclaration)
assert isinstance(initial.item.body, S.AttributedStatement) and isinstance(initial.item.body.statement, S.CaseStatement)
conditional = assign.assignments[0].value
assert conditional.attributes[0].specs[0].name.spelling == "x" and conditional.condition.attributes[0].specs[0].name.spelling == "mark"
calls = [s.value for s in block.body.items[:3]]
assert calls[0].attributes[0].specs[0].name.spelling == "inline" and calls[1].attributes[0].specs[0].name.spelling == "fast"
assert calls[2].attributes[0].specs[0].name.spelling == "neg"
body = block.body.items
assert body[3].expression.attributes and body[3].expression.postfix and body[4].expression.attributes
assert isinstance(body[5], S.LabeledStatement) and isinstance(body[5].statement, S.AttributedStatement)
assert isinstance(cls.items[0], S.AttributedItem) and isinstance(cls.items[0].item, S.VariableDeclaration)
assert same(read(SystemVerilog2023.print(unit)), unit)
print("ok")


## VLGRD-19 · Rise, fall and turn-off delays, min:typ:max, streaming, repeated patterns, tolerance ranges, empty arguments, `$root`, `$unit` and `type()`

In [ ]:
wire, assign, initial, instance, variable = items('''wire #(1, 2, 3) w; assign #(1:2:3, 4) w = x;
initial begin #(1:2:3) x = 1; $display(a,, b); end
u i(a, , b);
var type(a) v;''')
assert [wire.delay.value.spelling, wire.delay.fall.spelling, wire.delay.turnoff.spelling] == ["1", "2", "3"]
assert isinstance(assign.delay.value, S.MinTypMaxExpression) and assign.delay.fall.spelling == "4" and assign.delay.turnoff is None
timed, display = initial.body.items
assert isinstance(timed.timing.value.expression, S.MinTypMaxExpression) and timed.timing.fall is None
assert isinstance(display.expression.arguments[1], S.EmptyArgument)
assert [type(c).__name__ for c in instance.instances[0].connections] == ["NameExpression", "EmptyArgument", "NameExpression"]
assert isinstance(variable.type, S.TypeReference) and variable.type.operand.name.spelling == "a"
plain, sliced, typed, ranged = [expression(e) for e in ["{>>{a, b}}", "{<< 8 {a with [0 +: 2]}}", "{<<byte{a with [1]}}",
                                                         "{>> {b with [1:2]}}"]]
assert plain.operator == ">>" and plain.slice is None and [i.expression.name.spelling for i in plain.items] == ["a", "b"]
assert sliced.operator == "<<" and sliced.slice.spelling == "8" and sliced.items[0].operator == "+:"
assert isinstance(typed.slice, S.IntegerAtomType) and typed.items[0].left.spelling == "1" and typed.items[0].operator is None
assert ranged.items[0].operator == ":" and ranged.items[0].right.spelling == "2"
repeated = expression("'{2{a, b}}")
assert repeated.count.spelling == "2" and len(repeated.items) == 2 and repeated.type is None
inside = expression("a inside {[1 +/- 2], [3 +%- 4], [5:6]}")
assert [r.operator for r in inside.set] == ["+/-", "+%-", None]
call = expression("f(a, , b)")
assert isinstance(call.arguments[1], S.EmptyArgument)
root, unit_scoped, cast = expression("$root.top.a"), expression("$unit::a"), expression("type(a)'(b)")
assert isinstance(expression("{}"), S.EmptyQueue)
assert isinstance(root.value.value, S.RootExpression) and [root.value.member.spelling, root.member.spelling] == ["top", "a"]
assert isinstance(unit_scoped.name, S.ScopedName) and isinstance(unit_scoped.name.scope, S.UnitName)
assert isinstance(cast.type, S.TypeReference)
assert isinstance(items("var type(int) t;")[0].type.operand, S.DataType)
print("ok")


## VLGRD-20 · Strengths, `vectored` and `scalared`, net types, aliases, `defparam`, time units, enumerations' ranges, randomized members, `const ref` and `ref static`, and type restrictions

In [ ]:
declared = items('''wire (strong0, weak1) w1; trireg (small) t; wire (highz1, pull0) vectored [1:0] w2; tri scalared [3:0] w3 = 1;
assign (pull1, supply0) #1 a = b;
nettype logic [1:0] nt; nettype logic nt2 with p::f;
alias a = b = c[1:0];
defparam u.W = 2, v.X = 3;
typedef enum {A[2] = 0, B[1:2], C} e;
typedef struct { rand int a; randc bit [1:0] b; int c; } st;
function int f(const ref int a, ref static int b); endfunction''')
w1, t, w2, w3, assign, nt, nt2, alias, defparam, enum, struct, function = declared
assert (w1.strength.first, w1.strength.second, t.strength.size) == ("strong0", "weak1", "small")
assert w2.strength.first == "highz1" and w2.expansion == "vectored" and w3.expansion == "scalared" and w3.strength is None
assert assign.strength.second == "supply0" and assign.delay.value.spelling == "1"
assert nt.type.keyword == "logic" and nt.function is None and isinstance(nt2.function, S.ScopedName)
assert len(alias.nets) == 3 and isinstance(alias.nets[2], S.RangeSelect)
assert [str(SystemVerilog2023.print(a.target)) for a in defparam.assignments] == ["u.W", "v.X"]
assert [(m.name.spelling, m.left, m.right) for m in enum.type.members][2] == ("C", None, None)
assert enum.type.members[0].left.spelling == "2" and enum.type.members[0].value.spelling == "0"
assert (enum.type.members[1].left.spelling, enum.type.members[1].right.spelling) == ("1", "2")
assert [m.random for m in struct.type.members] == ["rand", "randc", None]
assert [(p.const, p.direction, p.static) for p in function.ports] == [(True, "ref", False), (False, "ref", True)]
units = read("package p; timeunit 1ns / 10ps; endpackage\ninterface i; timeunit 1ns; timeprecision 1ps; endinterface\n")
first, second, third = [*units.items[0].items, *units.items[1].items]
assert (first.keyword, first.time.spelling, first.precision.spelling) == ("timeunit", "1ns", "10ps")
assert second.precision is None and third.keyword == "timeprecision"
header = read("module m #(parameter type enum T = e, type interface class I = i) (); endmodule\n").items[0]
assert [p.restriction for p in header.parameters] == ["enum", "interface class"]
print("ok")


## VLGRD-21 · Non-ANSI ports of parts, concatenations, explicit names and none; explicit ANSI ports; modports' subroutines, clocking blocks and explicit ports

In [ ]:
unit = read("""module a (.p(x), .q(), .r({y, z}), , w[1:0], {u, v}, x[0]);
  input [1:0] x, w; input y, z, u, v;
endmodule
module b (input .p(x + 1), .q(y), output logic z, .r());
endmodule
interface i;
  modport mp (import f, g, export h, import task t(int a), import function int k(), clocking cb, input .a(b), c, output .d());
endinterface
""")
p, q, r, empty, part, concatenation, bit = unit.items[0].ports
assert (p.name.spelling, p.value.name.spelling, q.value) == ("p", "x", None)
assert [ref.name.spelling for ref in r.value.references] == ["y", "z"] and isinstance(empty, S.EmptyPort)
assert (part.left.spelling, part.operator, part.right.spelling) == ("1", ":", "0")
assert [ref.name.spelling for ref in concatenation.references] == ["u", "v"] and bit.left.spelling == "0" and bit.operator is None
ansi = unit.items[1].ports
assert [type(port).__name__ for port in ansi] == ["ExplicitAnsiPort", "ExplicitAnsiPort", "AnsiPort", "ExplicitAnsiPort"]
assert (ansi[0].direction, ansi[1].direction, ansi[1].value.name.spelling, ansi[3].value) == ("input", None, "y", None)
assert isinstance(ansi[0].value, S.BinaryExpression)
ports = unit.items[2].items[0].items[0].ports
assert [(port.keyword, port.name.spelling) for port in ports[:3]] == [("import", "f"), ("import", "g"), ("export", "h")]
assert isinstance(ports[3].prototype, S.TaskDeclaration) and ports[3].prototype.ports[0].name.spelling == "a"
assert isinstance(ports[4].prototype, S.FunctionDeclaration) and ports[4].name is None
assert isinstance(ports[5], S.ModportClocking) and ports[5].name.spelling == "cb"
assert [(port.direction, port.explicit, port.name.spelling) for port in ports[6:]] == [
    ("input", True, "a"), ("input", False, "c"), ("output", True, "d")]
assert ports[6].value.name.spelling == "b" and ports[8].value is None
print("ok")


## VLGRD-22 · `:final`, `:initial` and `:extends`, `extends base(default)`, `local typedef`, `;`, `typedef bus.t`, skipped `foreach` variables and `ifdef conditions

In [ ]:
unit = read("""class :final c extends b(default); ;
  local typedef int t; protected typedef struct { int a; int b; } s;
  virtual function :initial :final void f(); endfunction
  extern virtual function :extends void g();
  virtual task :final t2(); endtask
  constraint :initial k { x > 0; }
  extern constraint :extends :final k2;
  constraint k3 { foreach (q[, j, , k]) q[j] > 0; }
endclass
`ifdef (A && !B || (C -> D) || E <-> F)
module m; endmodule
`elsif (G)
`elsif H
`endif
module n (bus_if bus); typedef bus.data_t t; initial foreach (q[i, ]) x = 1; endmodule
""")
c = unit.items[0]
assert c.final and c.defaulted and c.base.name.spelling == "b" and not c.arguments
empty, t, s, f, g, t2, k, k2, k3 = c.items
assert isinstance(empty, S.EmptyItem) and (t.visibility, s.visibility) == ("local", "protected")
assert [(m.specifier, m.final) for m in (f, g, t2, k, k2, k3)] == [
    ("initial", True), ("extends", False), (None, True), ("initial", False), ("extends", True), (None, False)]
assert [type(v).__name__ for v in k3.items[0].variables] == ["EmptyArgument", "Identifier", "EmptyArgument", "Identifier"]
directive = unit.items[1]
assert directive.name is None and isinstance(directive.condition, S.ParenthesizedExpression)
top = directive.condition.expression
assert top.operator == "<->" and top.left.operator == "||" and top.left.left.left.right.operator == "!"
assert directive.branches[0].condition.expression.name.spelling == "G" and directive.branches[1].name.spelling == "H"
typedef, initial = unit.items[2].items
assert typedef.type.name.interface.spelling == "bus" and typedef.type.name.name.spelling == "data_t"
assert [type(v).__name__ for v in initial.body.variables] == ["Identifier", "EmptyArgument"]
print("ok")


## VLGRD-23 · `force`, `release`, `assign`, `deassign`, `wait fork`, `wait_order`, `->>` after a delay or an event, `repeat` event controls and elaboration tasks

In [ ]:
unit = read("""module m;
$error("bad %d", 1); $info; $fatal(1, , "x");
generate if (1) $warning("w"); endgenerate
initial begin force a[0] = 1; release a; assign b = 1; deassign b; wait fork;
  wait_order (a, b.c) x = 1; else y = 1; wait_order (a, b); wait_order (a) else y = 1;
  ->> #1 e; ->> @(posedge clk) e; ->> e; a <= repeat (2) @(posedge clk) b; end
endmodule
""")
error, info, fatal, generate, initial = unit.items[0].items
assert (error.name, [type(a).__name__ for a in error.arguments]) == ("$error", ["StringLiteral", "IntegerLiteral"])
assert info.arguments == [] and isinstance(fatal.arguments[1], S.EmptyArgument)
assert generate.items[0].consequence.name == "$warning"
force, release, assign, deassign, fork, ordered, plain, failing, delayed, evented, untimed, repeated = initial.body.items
assert [(s.keyword, type(s).__name__) for s in (force, release, assign, deassign)] == [
    ("force", "ForceStatement"), ("release", "ReleaseStatement"), ("assign", "ForceStatement"),
    ("deassign", "ReleaseStatement")]
assert isinstance(force.target, S.IndexExpression) and force.value.spelling == "1"
assert isinstance(fork, S.WaitForkStatement) and len(ordered.events) == 2 and isinstance(ordered.events[1], S.MemberExpression)
assert ordered.pass_action is not None and ordered.fail_action is not None
assert plain.pass_action is None and plain.fail_action is None  # `;`: no action
assert failing.pass_action is None and failing.fail_action is not None
assert isinstance(delayed.timing, S.DelayControl) and isinstance(evented.timing, S.EventControl) and untimed.timing is None
assert delayed.nonblocking and repeated.timing.count.spelling == "2"
assert repeated.timing.event.events[0].edge == "posedge"
print("ok")


## VLGRD-24 · DPI imports and exports, `bind`, `extern` modules, interfaces and programs, and `.*` ports

In [ ]:
unit = read("""module m;
import "DPI-C" function int f(int a);
import "DPI-C" context task t();
import "DPI-C" pure c_name = function int g();
import "DPI" function void h();
export "DPI-C" function f;
export "DPI-C" e = task t;
endmodule
bind m checker_unit c (.*);
bind m: u1, u2 mon #(1) i (.a(b)), j ();
bind top.m mon k ();
extern module e #(parameter W = 1) (input a);
extern interface ei (input a);
extern program ep;
module e (.*); endmodule
""")
f, t, g, h, export_f, export_t = unit.items[0].items
assert (f.spec, f.property, f.c_name, type(f.prototype).__name__) == ("DPI-C", None, None, "FunctionDeclaration")
assert f.prototype.ports[0].name.spelling == "a" and (t.property, type(t.prototype).__name__) == ("context", "TaskDeclaration")
assert (g.property, g.c_name.spelling, h.spec) == ("pure", "c_name", "DPI")
assert (export_f.keyword, export_f.name.spelling, export_f.c_name) == ("function", "f", None)
assert (export_t.keyword, export_t.c_name.spelling) == ("task", "e")
star, listed, hierarchical, extern_module, extern_interface, extern_program, defined = unit.items[1:]
assert star.target.name.spelling == "m" and star.instances == [] and isinstance(star.instantiation.instances[0].connections[0],
                                                                              S.WildcardConnection)
assert [i.name.spelling for i in listed.instances] == ["u1", "u2"] and len(listed.instantiation.instances) == 2
assert isinstance(hierarchical.target, S.MemberExpression)
assert extern_module.extern and extern_module.parameters[0].assignments[0].name.spelling == "W" and extern_module.items == []
assert isinstance(extern_interface, S.InterfaceDeclaration) and extern_interface.extern
assert isinstance(extern_program, S.ProgramDeclaration) and extern_program.ports == []
assert not defined.extern and [type(p).__name__ for p in defined.ports] == ["WildcardPort"]
print("ok")


## VLGRD-25 · `randsequence`: productions, ports, weights, code, `rand join`, `if`, `repeat` and `case`

In [ ]:
body = statements("""randsequence (main)
  main : first second(2) | third := 2 | rand join (0.5) first third | rand join second(1) first := 1 { x = 3; };
  first : { x = 1; // one
  } ;
  second(int n = 1) : if (a) third else first | repeat (2) third | case (b) 0, 1: first; default: second(3); endcase | if (c) first;
  void f : { return; };
endsequence
randsequence () a : b; b : { }; endsequence""")
sequence, unnamed = body
assert sequence.first.spelling == "main" and unnamed.first is None
main, first, second, f = sequence.productions
plain, weighted, joined, coded = main.rules
assert [i.name.spelling for i in plain.items] == ["first", "second"] and plain.items[1].arguments[0].spelling == "2"
assert weighted.weight.spelling == "2" and weighted.code is None and not weighted.rand_join
assert joined.rand_join and joined.bias.spelling == "0.5" and joined.weight is None
assert coded.rand_join and coded.bias is None and coded.weight.spelling == "1"
assert isinstance(coded.code.items[0], S.AssignmentStatement)
assert isinstance(first.rules[0].items[0], S.ProductionCode) and isinstance(first.rules[0].items[0].items[1], S.Comment)
assert second.ports[0].name.spelling == "n" and second.ports[0].value.spelling == "1"
choice, repeat, case, bare = [r.items[0] for r in second.rules]
assert bare.alternative is None
assert (choice.condition.name.spelling, choice.consequence.name.spelling, choice.alternative.name.spelling) == ("a", "third",
                                                                                                             "first")
assert repeat.count.spelling == "2" and repeat.item.name.spelling == "third"
assert [[v.spelling for v in i.values] for i in case.items] == [["0", "1"], []] and case.items[1].item.arguments
assert f.type.keyword == "void" and isinstance(f.rules[0].items[0].items[0], S.ReturnStatement)
assert unnamed.productions[1].rules[0].items[0].items == []
print("ok")


## VLGRD-26 · Checkers: ports, `rand` variables, nested checkers and instances among items and in procedures; package exports; nets of user-defined types

In [ ]:
unit = read("""package p; checker pc (a); assert property (a); endchecker export q::*; export q::x, r::y; export *::*; endpackage
checker c (input logic clk, a, output bit b = 0, property pr, sequence s = a ##1 a, untyped u, event e = $inferred_clock);
  default clocking @(posedge clk); endclocking
  rand bit r;
  rand const int k;
  bit v = 0;
  checker inner; endchecker
endchecker : c
module m;
  nettype logic nt; nt #1 w, v = 1; nt w2;
  c c1 (.clk(clk), .a(x));
  p::pc c4 (x);
  always @(posedge clk) begin c c2 (clk, x), c3 (.*); p::pc c5 (x); end
endmodule
""")
package, checker, module = unit.items
pc, wildcard, named, everything = package.items
assert isinstance(pc, S.CheckerDeclaration) and pc.ports[0].name.spelling == "a" and not pc.labeled
assert wildcard.items[0].name is None and [i.name.spelling for i in named.items] == ["x", "y"] and everything.all
assert checker.labeled and [p.direction for p in checker.ports[:3]] == ["input", None, "output"]
assert checker.ports[2].value.spelling == "0" and isinstance(checker.ports[4].type, S.KeywordType)
clocking, r, k, v, inner = checker.items
assert (r.random, k.random, k.const, v.random) == ("rand", "rand", True, None) and isinstance(inner, S.CheckerDeclaration)
_, user, variable, c1, c4, always = module.items
assert user.net_type is None and user.type.name.spelling == "nt" and user.delay.value.spelling == "1"
assert isinstance(variable, S.VariableDeclaration) and variable.type.name.spelling == "nt"
assert c1.module.spelling == "c" and isinstance(c4.module, S.ScopedName)
c2, c5 = always.body.body.items
assert isinstance(c2, S.CheckerStatement) and [i.name.spelling for i in c2.instantiation.instances] == ["c2", "c3"]
assert isinstance(c5.instantiation.module, S.ScopedName)
print("ok")


## VLGRD-27 · Tagged and soft unions, tagged values, `matches` in `case`, `if` and `?:`, `&&&`, and patterns of every kind

In [ ]:
declared = items("""typedef union tagged packed { int a; void b; } t;
typedef union soft { int a; bit [31:0] c; } s;
initial begin
  x = tagged a 5; x = tagged b; y = (tagged a 5) + 1;
  case (x) matches tagged a .v &&& v > 0: ; tagged b: ; .*: ; default: ; endcase
  if (x matches tagged a .v &&& v > 1) ;
  if (x matches (tagged a 3)) ;
  y = x matches tagged a .v &&& v > 1 ? v : 0;
  case (p) matches '{.q, 2}: ; '{f: .r, g: tagged a .*}: ; endcase
  if (a &&& b) ;
end""")
tagged_union, soft_union, initial = declared
assert (tagged_union.type.qualifier, tagged_union.type.packed, soft_union.type.qualifier) == ("tagged", True, "soft")
assert tagged_union.type.members[1].type.keyword == "void"
five, bare, parenthesized, case, guarded, constant, conditional, structures, plain = initial.body.items
assert (five.value.member.spelling, five.value.value.spelling, bare.value.value) == ("a", "5", None)
assert isinstance(parenthesized.value.left, S.ParenthesizedExpression)
assert case.matches and [type(i).__name__ for i in case.items] == ["PatternCaseItem"] * 3 + ["CaseItem"]
first = case.items[0]
assert first.pattern.member.spelling == "a" and first.pattern.pattern.name.spelling == "v" and first.guard.operator == ">"
assert case.items[1].pattern.pattern is None and isinstance(case.items[2].pattern, S.WildcardPattern)
assert isinstance(guarded.condition, S.PredicateExpression) and isinstance(guarded.condition.conditions[0], S.MatchesExpression)
assert constant.condition.pattern.pattern.spelling == "3"  # the pattern's parentheses are not kept
assert isinstance(conditional.value.condition, S.PredicateExpression)
ordered, named = [i.pattern for i in structures.items]
assert isinstance(ordered.items[0], S.VariablePattern) and ordered.items[1].spelling == "2"
assert [m.name.spelling for m in named.items] == ["f", "g"] and isinstance(named.items[1].pattern.pattern, S.WildcardPattern)
assert [c.name.spelling for c in plain.condition.conditions] == ["a", "b"]
print("ok")


## VLGRD-28 · Gate instances, pulls, user-defined primitives with ANSI and non-ANSI ports, and their tables and instances

In [ ]:
unit = read("""primitive mux (output o, input s, a, b);
  table
    // s a b : o
    0 1 ? : 1 ;
    1 ? 0 : 0 ;
  endtable
endprimitive : mux
primitive latch (q, d, en);
  output q; reg q; input d, en;
  initial q = 1'b0;
  table
    (01) 1 : ? : 1 ;
    r ? : 0 : - ;
  endtable
endprimitive
primitive dff (output reg q = 0, input d, clk); table 0 r : ? : 0 ; endtable endprimitive
module m;
  and #(1, 2) g1 (o, a, b), g2 (o2, a, b);
  nand (strong0, weak1) (o3, a, b);
  bufif0 #3 b1 [1:0] (o4, a, en);
  pulldown (weak0) pd (q);
  latch #1 (q, d, en);
endmodule
""")
mux, latch, dff, module = unit.items
assert mux.labeled and [(p.direction, [n.spelling for n in p.names]) for p in mux.ports] == [("output", ["o"]),
                                                                                                ("input", ["s", "a", "b"])]
assert isinstance(mux.entries[0], S.Comment) and (mux.entries[1].inputs, mux.entries[1].current, mux.entries[1].output) == (
    "0 1 ?", None, "1")
assert [p.spelling for p in latch.ports] == ["q", "d", "en"]
assert [(d.direction, d.reg) for d in latch.declarations] == [("output", False), (None, True), ("input", False)]
assert latch.initial.name.spelling == "q" and latch.entries[0].inputs == "(01) 1" and latch.entries[1].output == "-"
assert dff.ports[0].reg and dff.ports[0].value.spelling == "0"
gates, nand, buffers, pulldown, udp = module.items
assert gates.keyword == "and" and gates.delay.fall.spelling == "2" and [i.name.spelling for i in gates.instances] == ["g1", "g2"]
assert nand.strength.second == "weak1" and nand.instances[0].name is None and len(nand.instances[0].terminals) == 3
assert buffers.instances[0].dimensions[0].left.spelling == "1"
assert isinstance(pulldown.strength, S.PullStrength) and pulldown.strength.strength == "weak0"
assert udp.keyword is None and udp.primitive.spelling == "latch" and udp.delay.value.spelling == "1"
print("ok")


## VLGRD-29 · Specify blocks: specify parameters, module paths of every form, timing checks and pulse styles

In [ ]:
specify, specparam = items("""specify
  // the delays of the paths
  specparam [7:0] w = 8, PATHPULSE$a$y = (1, 2);
  (a => y) = (tRise, tFall);
  (a, b *> y, q) = 3;
  (a +=> y) = 1;
  (a -*> y) = 1;
  if (b) (a => y) = 1;
  ifnone (a => y) = 2;
  (posedge clk => (q +: d)) = (1, 2);
  (clk => (q : d)) = 2;
  $hold(posedge clk &&& en, d, 1, notifier);
  $setuphold(posedge clk, d, 1, 2, , , , dclk, dd);
  $period(edge [01, 10] clk, 10);
  pulsestyle_ondetect y, q;
endspecify
specparam s = 1:2:3;""")
comment, widths, parallel, full, plus, minus, conditional, ifnone, edged, colon, hold, setuphold, period, pulse = specify.items
assert isinstance(comment, S.Comment) and widths.type.dimensions[0].left.spelling == "7"
assert [(a.name.spelling, a.limit.spelling if a.limit is not None else None) for a in widths.assignments] == [
    ("w", None), ("PATHPULSE$a$y", "2")]
assert (parallel.operator, len(parallel.delays), full.operator, len(full.inputs), len(full.outputs)) == ("=>", 2, "*>", 2, 2)
assert (plus.polarity, plus.operator, minus.polarity, minus.operator) == ("+", "=>", "-", "*>")
assert conditional.condition.name.spelling == "b" and ifnone.ifnone and ifnone.condition is None
assert (edged.edge, edged.data_polarity, edged.data.name.spelling) == ("posedge", "+:", "d") and colon.data_polarity == ":"
event = hold.arguments[0]
assert (event.edge, event.terminal.name.spelling, event.condition.name.spelling) == ("posedge", "clk", "en")
assert [type(a).__name__ for a in setuphold.arguments][4:7] == ["EmptyArgument"] * 3
assert (period.arguments[0].edge, period.arguments[0].descriptors) == ("edge", "01, 10")
assert pulse.keyword == "pulsestyle_ondetect" and len(pulse.outputs) == 2
assert isinstance(specparam.assignments[0].value, S.MinTypMaxExpression)
print("ok")


## VLGRD-30 · `covergroup extends`, `@@` events, `matches` in selects, `super.new(default)`, non-ANSI interface ports, attributes on members, ports, connections, coverage and clocking items, and sequences as arguments

In [ ]:
unit = read("""class b; covergroup g; coverpoint x; endgroup endclass
class d extends b;
  covergroup extends g; coverpoint y; endgroup
  function new(); super.new(default); endfunction
endclass
module m (a);
  bus_if.mp a;
  typedef struct { (* keep *) int x; rand bit y; } t;
  function int f((* unused *) int x, input int y); endfunction
  u i1 ((* a *) .x(y), (* b *) z);
  property p(sequence s); s; endproperty
  sequence q(s); s; endsequence
  assert property (p(a ##1 b));
  assert property (q(a[*2]));
  assert property (p(.s(a ##1 b)));
  clocking cb @(posedge clk); input x; (* prop *) property pc; x; endproperty endclocking
  covergroup g2 @@(begin f or end t.run);
    (* point *) coverpoint x { (* bin *) bins low = {0}; }
    x: cross a, b { (* sel *) bins s = x matches 2; bins w = binsof(a) with (a > 1) matches 0;
      (* helper *) function int h(); return 1; endfunction }
  endgroup
endmodule
""")
extended = unit.items[1].items[0]
assert extended.extends and extended.name.spelling == "g" and extended.ports == []
constructor = unit.items[1].items[1].body[0].expression
assert constructor.defaulted and isinstance(constructor.scope, S.SuperExpression) and constructor.arguments == []
items = unit.items[2].items
port, typedef, function, instance = items[:4]
assert (port.interface.spelling, port.modport.spelling, port.declarators[0].name.spelling) == ("bus_if", "mp", "a")
assert typedef.type.members[0].attributes[0].specs[0].name.spelling == "keep" and typedef.type.members[1].attributes == []
assert function.ports[0].attributes[0].specs[0].name.spelling == "unused" and function.ports[1].attributes == []
named, ordered = instance.instances[0].connections
assert isinstance(named, S.AttributedConnection) and isinstance(named.connection, S.NamedConnection)
assert isinstance(ordered.connection, S.NameExpression)
delayed, repeated, by_name = [i.assertion.spec.property for i in items[6:9]]
assert isinstance(delayed.arguments[0], S.DelaySequence) and isinstance(repeated.arguments[0], S.RepetitionSequence)
assert isinstance(by_name.arguments[0].value, S.DelaySequence)
clocking = items[9]
assert isinstance(clocking.items[1], S.AttributedItem) and isinstance(clocking.items[1].item, S.PropertyDeclaration)
group = items[10]
assert [(e.keyword, type(e.name).__name__) for e in group.clock.events] == [("begin", "NameExpression"),
                                                                            ("end", "MemberExpression")]
point, cross = group.items
assert isinstance(point, S.AttributedItem) and isinstance(point.item.items[0], S.AttributedItem)
selection, filtered, helper = cross.body
assert isinstance(selection.item.select, S.MatchesBinsSelect) and selection.item.select.count.spelling == "2"
assert isinstance(filtered.select.select, S.FilteredBinsSelect) and filtered.select.count.spelling == "0"
assert isinstance(helper, S.AttributedItem) and isinstance(helper.item, S.FunctionDeclaration)
print("ok")
